# Module 01 — Classic CNN Architectures (SOLUTIONS)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Solution — Pre-Activation Residual Block

In [ ]:
class PreActivationBlock(nn.Module):
    """Pre-activation residual block (He et al., 2016).
    Order: BN → ReLU → Conv → BN → ReLU → Conv + shortcut.
    """
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-activation: BN → ReLU before convolution
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        return out + self.shortcut(x)


# Verify
block = PreActivationBlock(64, 64)
x = torch.randn(2, 64, 8, 8)
out = block(x)
print('Output shape:', out.shape)  # torch.Size([2, 64, 8, 8])

# With stride 2 (downsampling)
block_ds = PreActivationBlock(64, 128, stride=2)
out_ds = block_ds(x)
print('Downsampled shape:', out_ds.shape)  # torch.Size([2, 128, 4, 4])